# Selector CatBoost SHAP Review

This notebook compares feature selection modules in the workflow below:

1. Select top features per order/method.
2. Train a fresh CatBoost model using only each selected feature set.
3. Rank features by `abs(mean SHAP Bad - mean SHAP Good)`.
4. Display the top SHAP-delta table and saved x-y plots.

In [ ]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "examples"))

DATA_DIR = PROJECT_ROOT / "data" / "toy_semiconductor_wide"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "wide_toy_selector_comparison"

print("project:", PROJECT_ROOT)
print("data:", DATA_DIR)
print("output:", OUTPUT_DIR)
print("catboost installed:", importlib.util.find_spec("catboost") is not None)
print("matplotlib installed:", importlib.util.find_spec("matplotlib") is not None)

## 0. Package Check

If CatBoost or matplotlib is missing, run the next cell once. Then restart the notebook kernel.

In [ ]:
# Uncomment and run if needed, then restart the kernel.
# %pip install catboost matplotlib

## 1. Settings

For a quick first run, use `ORDERS = [1]`, `ROWS_PER_ORDER = 600`, and `FEATURES_PER_ORDER = 1000`. For the larger default toyset, use the values below.

In [ ]:
ORDERS = [1]
ROWS_PER_ORDER = 600
FEATURES_PER_ORDER = 1000
BOOSTER_GOOD_ROWS = 80
BOOSTER_BAD_ROWS = 30

SELECTED_FEATURES_PER_ORDER = 20
SHAP_TOP_N = 15
STABILITY_ROUNDS = 2
REGENERATE_DATA = True

METHODS = (
    "nonparametric_random",
    "distance",
    "catboost_shap_gap",
    "stability_consensus",
)

## 2. Generate Data

In [ ]:
from generate_wide_toy_semiconductor_dataset import generate_dataset

summary = generate_dataset(
    output_dir=DATA_DIR,
    orders=max(ORDERS),
    rows=ROWS_PER_ORDER,
    features_per_order=FEATURES_PER_ORDER,
    booster_good_rows=BOOSTER_GOOD_ROWS,
    booster_bad_rows=BOOSTER_BAD_ROWS,
    overwrite=REGENERATE_DATA,
)
summary

## 3. Run Selector Comparison and CatBoost SHAP Post-Evaluation

This cell can take a while. If CatBoost is not installed, the result files will contain `catboost_not_installed` warnings instead of SHAP values.

In [ ]:
from run_selector_comparison_on_wide_toyset import run_comparison

comparison, overlap, shap_delta, model_metrics = run_comparison(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    order_list=ORDERS,
    top_k=SELECTED_FEATURES_PER_ORDER,
    shap_top_n=SHAP_TOP_N,
    methods=METHODS,
    include_evidence_booster=True,
    run_catboost_eval=True,
    stability_rounds=STABILITY_ROUNDS,
)

comparison.head(20)

## 4. Model Metrics by Method

In [ ]:
model_metrics

## 5. SHAP Delta Top Features

The score below is `abs(mean SHAP in Bad - mean SHAP in Good)` from a fresh CatBoost model trained on each method's selected feature set.

In [ ]:
import pandas as pd

shap_top_path = OUTPUT_DIR / "catboost_post_eval" / "catboost_shap_delta_top_features.csv"
if shap_top_path.exists():
    shap_top = pd.read_csv(shap_top_path)
else:
    shap_top = shap_delta[shap_delta["shap_rank"] <= SHAP_TOP_N].copy() if not shap_delta.empty else pd.DataFrame()

display_cols = [
    "order_id",
    "method",
    "shap_rank",
    "feature_name",
    "shap_delta_abs",
    "shap_bad_mean",
    "shap_good_mean",
    "model_auc_train",
    "evaluation_warning",
]
existing = [col for col in display_cols if col in shap_top.columns]
shap_top[existing].head(80) if existing else shap_top.head(80)

## 6. Display Saved X-Y Plots

Each PNG shows full-pool wafer data: Ignored, Good, and Bad points are plotted together.

In [ ]:
from IPython.display import Image, display

plot_dir = OUTPUT_DIR / "catboost_post_eval" / "plots"
plot_paths = sorted(plot_dir.glob("*.png")) if plot_dir.exists() else []
print("plot count:", len(plot_paths))

for path in plot_paths:
    print(path.name)
    display(Image(filename=str(path)))

## 7. Method Overlap

This is not the main performance measure, but it helps show whether methods are choosing similar feature sets.

In [ ]:
overlap.head(50)

## 8. Useful Output Files

In [ ]:
for path in [
    OUTPUT_DIR / "combined_selector_comparison.csv",
    OUTPUT_DIR / "method_overlap_jaccard.csv",
    OUTPUT_DIR / "catboost_post_eval" / "catboost_shap_delta_top_features.csv",
    OUTPUT_DIR / "catboost_post_eval" / "catboost_model_metrics_by_method.csv",
    OUTPUT_DIR / "catboost_post_eval" / "plots",
]:
    print(path, "exists=", path.exists())